In [0]:
#Carregano Tabelas do Unity Catalog - Transações de Cartão de Crédito

df_creditcard = spark.table('workspace.projeto_creditcard.creditcard').toPandas()

df_classe = spark.table('workspace.projeto_creditcard.descricao_classe').toPandas()

df_aporte = spark.table('workspace.projeto_creditcard.descricao_aporte').toPandas()

In [0]:
# Vizualição amostral das bases

print("=== CREDITCARD ===")
display(df_creditcard.head())

print("\n=== DESCRIÇÃO CLASSE ===")
display(df_classe.head())

print("\n=== DESCRIÇÃO APORTE ===")
display(df_aporte.head())

In [0]:
# Preparação de views para manipulação de dados via Spark

import time

# Criar views temporárias para usar Spark SQL
spark.createDataFrame(df_creditcard).createOrReplaceTempView("vw_creditcard")
spark.createDataFrame(df_classe).createOrReplaceTempView("vw_classe")
spark.createDataFrame(df_aporte).createOrReplaceTempView("vw_aporte")

# Função para medir tempo de execução
def medir_tempo(nome, funcao):
    inicio = time.time()
    resultado = funcao()
    fim = time.time()
    tempo = (fim - inicio) * 1000  # em milissegundos
    print(f"{nome}: {tempo:.2f}ms")
    return resultado, tempo

print("✅ Ambiente preparado! Views temporárias criadas.")

In [0]:
# JOIN utilizando Pyspark API

print("=" * 60)
print("JOIN SIMPLES - Creditcard + Descrição Classe")
print("=" * 60)

# Método 1: PySpark API
def join_pyspark():
    df_spark_cc = spark.createDataFrame(df_creditcard)
    df_spark_classe = spark.createDataFrame(df_classe)
    
    resultado = df_spark_cc.join(
        df_spark_classe, 
        df_spark_cc['Class'] == df_spark_classe['class'], 
        'left'
    ).select(
        'Time', 'Amount', 'descricao',
        df_spark_cc['Class']
    )
    return resultado.limit(5).toPandas()

resultado_pyspark, tempo_pyspark = medir_tempo("PySpark API", join_pyspark)
print("\nResultado PySpark:")
display(resultado_pyspark)

In [0]:
# JOIN utilizando SQL diratamente no Spark

def join_sql():
    query = """
    SELECT 
        cc.Time,
        cc.Amount,
        cc.Class,
        dc.descricao
    FROM vw_creditcard cc
    LEFT JOIN vw_classe dc ON cc.Class = dc.class
    LIMIT 5
    """
    return spark.sql(query).toPandas()

resultado_sql, tempo_sql = medir_tempo("Spark SQL", join_sql)
print("\nResultado Spark SQL:")
display(resultado_sql)

# Comparação
print(f"\n📊 COMPARAÇÃO:")
print(f"PySpark API: {tempo_pyspark:.2f}ms")
print(f"Spark SQL:   {tempo_sql:.2f}ms")
diferenca = abs(tempo_pyspark - tempo_sql)
mais_rapido = "PySpark API" if tempo_pyspark < tempo_sql else "Spark SQL"
print(f"Diferença: {diferenca:.2f}ms - {mais_rapido} foi mais rápido")

In [0]:
# CASE WHEN com Spark

print("=" * 60)
print("JOIN COM LÓGICA - Classificar Amount em Faixas")
print("=" * 60)

# Método 1: PySpark API
from pyspark.sql import functions as F

def classificar_pyspark():
    df_spark_cc = spark.createDataFrame(df_creditcard)
    
    resultado = df_spark_cc.withColumn(
        'faixa_aporte',
        F.when(F.col('Amount') <= 50, 'Aporte Muito Baixo')
         .when((F.col('Amount') > 50) & (F.col('Amount') <= 100), 'Aporte Baixo')
         .when((F.col('Amount') > 100) & (F.col('Amount') <= 300), 'Aporte Médio')
         .when((F.col('Amount') > 300) & (F.col('Amount') <= 1000), 'Aporte Alto')
         .otherwise('Sem Aporte')
    ).select('Time', 'Amount', 'Class', 'faixa_aporte')
    
    return resultado.limit(10).toPandas()

resultado_pyspark2, tempo_pyspark2 = medir_tempo("PySpark API (CASE WHEN)", classificar_pyspark)
print("\nResultado PySpark:")
display(resultado_pyspark2)

In [0]:
# CASE WHEN com SQL via Spark

def classificar_sql():
    query = """
    SELECT 
        Time,
        Amount,
        Class,
        CASE 
            WHEN Amount <= 50 THEN 'Aporte Muito Baixo'
            WHEN Amount > 50 AND Amount <= 100 THEN 'Aporte Baixo'
            WHEN Amount > 100 AND Amount <= 300 THEN 'Aporte Médio'
            WHEN Amount > 300 AND Amount <= 1000 THEN 'Aporte Alto'
            ELSE 'Sem Aporte'
        END as faixa_aporte
    FROM vw_creditcard
    LIMIT 10
    """
    return spark.sql(query).toPandas()

resultado_sql2, tempo_sql2 = medir_tempo("Spark SQL (CASE WHEN)", classificar_sql)
print("\nResultado Spark SQL:")
display(resultado_sql2)

# Comparação
print(f"\n📊 COMPARAÇÃO:")
print(f"PySpark API: {tempo_pyspark2:.2f}ms")
print(f"Spark SQL:   {tempo_sql2:.2f}ms")
diferenca = abs(tempo_pyspark2 - tempo_sql2)
mais_rapido = "PySpark API" if tempo_pyspark2 < tempo_sql2 else "Spark SQL"
print(f"Diferença: {diferenca:.2f}ms - {mais_rapido} foi mais rápido")

In [0]:
# GROUP BY com Spark

print("=" * 60)
print("AGREGAÇÃO - Estatísticas por Classe")
print("=" * 60)

def agregar_pyspark():
    df_spark_cc = spark.createDataFrame(df_creditcard)
    
    resultado = df_spark_cc.groupBy('Class').agg(
        F.count('*').alias('total_transacoes'),
        F.sum('Amount').alias('valor_total'),
        F.avg('Amount').alias('valor_medio'),
        F.min('Amount').alias('valor_minimo'),
        F.max('Amount').alias('valor_maximo')
    ).orderBy('Class')
    
    return resultado.toPandas()

resultado_pyspark3, tempo_pyspark3 = medir_tempo("PySpark API (GROUP BY)", agregar_pyspark)
print("\nResultado PySpark:")
display(resultado_pyspark3)

In [0]:
# GROUP BY com SQL via Spark

def agregar_sql():
    query = """
    SELECT 
        Class,
        COUNT(*) as total_transacoes,
        SUM(Amount) as valor_total,
        AVG(Amount) as valor_medio,
        MIN(Amount) as valor_minimo,
        MAX(Amount) as valor_maximo
    FROM vw_creditcard
    GROUP BY Class
    ORDER BY Class
    """
    return spark.sql(query).toPandas()

resultado_sql3, tempo_sql3 = medir_tempo("Spark SQL (GROUP BY)", agregar_sql)
print("\nResultado Spark SQL:")
display(resultado_sql3)

# Comparação
print(f"\n📊 COMPARAÇÃO:")
print(f"PySpark API: {tempo_pyspark3:.2f}ms")
print(f"Spark SQL:   {tempo_sql3:.2f}ms")
diferenca = abs(tempo_pyspark3 - tempo_sql3)
mais_rapido = "PySpark API" if tempo_pyspark3 < tempo_sql3 else "Spark SQL"
print(f"Diferença: {diferenca:.2f}ms - {mais_rapido} foi mais rápido")

In [0]:
# Rankear valores com Spark


print("=" * 60)
print("WINDOW FUNCTIONS - Ranking por Classe")
print("=" * 60)

from pyspark.sql.window import Window

def window_pyspark():
    df_spark_cc = spark.createDataFrame(df_creditcard)
    
    # Definir a janela: particionar por Class, ordenar por Amount decrescente
    janela = Window.partitionBy('Class').orderBy(F.desc('Amount'))
    
    resultado = df_spark_cc.withColumn(
        'ranking', 
        F.row_number().over(janela)
    ).withColumn(
        'valor_acumulado',
        F.sum('Amount').over(janela.rowsBetween(Window.unboundedPreceding, Window.currentRow))
    ).filter(
        F.col('ranking') <= 3  # Pegar apenas top 3 de cada classe
    ).select(
        'Class', 'Amount', 'Time', 'ranking', 'valor_acumulado'
    ).orderBy('Class', 'ranking')
    
    return resultado.toPandas()

resultado_pyspark4, tempo_pyspark4 = medir_tempo("PySpark API (WINDOW)", window_pyspark)
print("\nResultado PySpark - Top 3 por classe:")
display(resultado_pyspark4)

In [0]:
# Rankear valores com SQL via Spark


def window_sql():
    query = """
    SELECT 
        Class,
        Amount,
        Time,
        ranking,
        valor_acumulado
    FROM (
        SELECT 
            Class,
            Amount,
            Time,
            ROW_NUMBER() OVER (PARTITION BY Class ORDER BY Amount DESC) as ranking,
            SUM(Amount) OVER (PARTITION BY Class ORDER BY Amount DESC 
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as valor_acumulado
        FROM vw_creditcard
    ) ranked
    WHERE ranking <= 3
    ORDER BY Class, ranking
    """
    return spark.sql(query).toPandas()

resultado_sql4, tempo_sql4 = medir_tempo("Spark SQL (WINDOW)", window_sql)
print("\nResultado Spark SQL - Top 3 por classe:")
display(resultado_sql4)

# Comparação
print(f"\n📊 COMPARAÇÃO:")
print(f"PySpark API: {tempo_pyspark4:.2f}ms")
print(f"Spark SQL:   {tempo_sql4:.2f}ms")
diferenca = abs(tempo_pyspark4 - tempo_sql4)
mais_rapido = "PySpark API" if tempo_pyspark4 < tempo_sql4 else "Spark SQL"
print(f"Diferença: {diferenca:.2f}ms - {mais_rapido} foi mais rápido")

In [0]:
# Resumo Final de Time Spark vs SQL via Spark

print("=" * 60)
print("RESUMO COMPARATIVO - PySpark API vs Spark SQL")
print("=" * 60)

resumo = {
    'Operação': [
        'JOIN Simples',
        'JOIN + CASE WHEN',
        'Agregação GROUP BY',
        'Window Functions (Ranking)'
    ],
    'PySpark API (ms)': [tempo_pyspark, tempo_pyspark2, tempo_pyspark3, tempo_pyspark4],
    'Spark SQL (ms)': [tempo_sql, tempo_sql2, tempo_sql3, tempo_sql4]
}

import pandas as pd
df_resumo = pd.DataFrame(resumo)
df_resumo['Diferença (ms)'] = abs(df_resumo['PySpark API (ms)'] - df_resumo['Spark SQL (ms)'])
df_resumo['Mais Rápido'] = df_resumo.apply(
    lambda row: 'PySpark API' if row['PySpark API (ms)'] < row['Spark SQL (ms)'] else 'Spark SQL', 
    axis=1
)

display(df_resumo)

print("\n💡 OBSERVAÇÕES:")
print("• Em bases pequenas, a diferença é mínima (overhead de parsing)")
print("• Em produção (grandes volumes), Spark SQL costuma ser otimizado pelo Catalyst")
print("• PySpark API oferece mais flexibilidade e type-safety")
print("• Window Functions são essenciais para análises avançadas (ranking, cumulativos)")
print("• Ambos geram o mesmo plano de execução físico no final")